# MovieLens 推荐系统和聚类分析 - 快速演示

这个 notebook 提供了项目的快速演示和交互式分析。

## 1. 导入库和设置

In [ ]:
import sys
sys.path.append('..')

from src.data_loader import MovieLensLoader
from src.recommender import SVDRecommender
from src.clustering import MovieClusterer, DimensionalityReducer
from src.visualization import Visualizer

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

import warnings
warnings.filterwarnings('ignore')

## 2. 加载数据

In [ ]:
# 加载数据（使用采样加快速度）
loader = MovieLensLoader(data_dir='../data')
loader.load_data(sample_size=500000)  # 使用 50 万条数据进行演示

# 查看数据概览
print("\n评分数据示例：")
display(loader.ratings.head())

print("\n电影数据示例：")
display(loader.movies.head())

# 数据统计
print(f"\n数据统计：")
print(f"评分数量: {len(loader.ratings):,}")
print(f"用户数量: {loader.ratings['userId'].nunique():,}")
print(f"电影数量: {loader.ratings['movieId'].nunique():,}")

## 3. 推荐系统

In [ ]:
# 预处理数据
user_movie_matrix, filtered_ratings = loader.preprocess_for_recommendation(
    min_user_ratings=30,
    min_movie_ratings=30
)

# 训练推荐模型
recommender = SVDRecommender(n_components=30)
recommender.fit(user_movie_matrix)

In [ ]:
# 为随机用户生成推荐
sample_user = user_movie_matrix.index[10]
print(f"为用户 {sample_user} 推荐的电影：\n")

recommendations = recommender.recommend_for_user(sample_user, top_n=10)

for i, (movie_id, score) in enumerate(recommendations, 1):
    movie_info = loader.get_movie_info([movie_id])
    if len(movie_info) > 0:
        title = movie_info.iloc[0]['title']
        genres = movie_info.iloc[0]['genres']
        print(f"{i}. {title}")
        print(f"   类型: {genres}")
        print(f"   预测评分: {score:.2f}\n")

## 4. 聚类分析

In [ ]:
# 创建电影特征
movie_features = loader.create_movie_features(use_genome=False)
print(f"特征矩阵形状: {movie_features.shape}")

In [ ]:
# K-means 聚类
clusterer = MovieClusterer()
labels = clusterer.kmeans_clustering(movie_features, n_clusters=8)

# 查看聚类统计
cluster_stats = clusterer.get_cluster_statistics(loader.movies)
display(cluster_stats)

In [ ]:
# 查看每个聚类的电影示例
for cluster_id in range(min(3, clusterer.n_clusters)):
    print(f"\n聚类 {cluster_id} 的电影示例：")
    cluster_movies = clusterer.movie_ids[labels == cluster_id][:5]
    cluster_info = loader.get_movie_info(cluster_movies)
    for _, movie in cluster_info.iterrows():
        print(f"  - {movie['title']} ({movie['genres']})")

## 5. 降维分析

In [ ]:
# 比较降维前后的聚类效果
reducer = DimensionalityReducer(n_components=10)
comparison = reducer.compare_clustering_with_without_pca(
    movie_features,
    n_clusters=8,
    method='kmeans'
)

In [ ]:
# 可视化 PCA 方差解释
pca = comparison['pca']
cumsum = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Variance Explained by Each PC')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumsum) + 1), cumsum, 'ro-')
plt.axhline(y=0.90, color='g', linestyle='--', label='90% Variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Variance Explained')
plt.legend()

plt.tight_layout()
plt.show()

## 6. 结论

在这个 notebook 中，我们展示了：

1. **推荐系统**：使用 SVD 为用户生成个性化推荐
2. **聚类分析**：将电影按特征分组，发现相似电影
3. **降维分析**：使用 PCA 减少特征维度，提高效率

完整的分析结果请运行 `python main.py`。